# SmolVLA × LIBERO: малобюджетная адаптация

Один последовательный запуск: seen-претрен → проверки корректности → K=0/язык → baseline → H1–H4 → таблицы результатов.

Критические правила:

- `libero_90` — только seen-часть;
- целевые задачи `libero_goal` выбираются **по тексту инструкции**;
- K=5/10/25 — строго первые K демонстраций;
- демонстрации других целевых задач не используются;
- никаких адаптеров: baseline и H1–H4 используют штатно обучаемые параметры SmolVLA;
- выбран Bonus A; при построении H1 значения действий удаляются до препроцессинга и не участвуют в представлениях, целях или кластеризации;
- если воспроизведение демонстрации не даёт success=1 или проверка seen-чекпойнта даёт 0/15, дорогая часть останавливается.


## Предсказания, зафиксированные до запусков

1. Результат K=0 после seen-претренинга на новых задачах `libero_goal` может быть очень низким, включая 0. Это не поломка, если воспроизведение демонстраций и проверка seen-задач проходят.
2. Baseline в среднем должен улучшаться 5 → 10 → 25 демонстраций; максимальный простор для сдвига кривой — K=5/10.
3. **H1, видео-динамика:** структура визуальных переходов seen-видео без использования действий даст полезный дополнительный сигнал; ожидаемый максимум эффекта при K=5.
4. **H2, инструкция:** правильная инструкция должна выделять в скрытых токенах представление, лучше согласованное с целевым действием, чем нерелевантная seen-инструкция; ожидаемый эффект K=5/10.
5. **H3, L2-SP:** удержание параметров около seen-чекпойнта должно уменьшать переобучение на малом числе демонстраций; ожидаемый эффект прежде всего K=5, возможное ухудшение при K=25.
6. **H4, повтор seen-данных:** один seen BC-батч каждые четыре целевых шага должен удерживать общие навыки; ожидаемый эффект K=5/10, вычислительная цена около +25%.
7. Если `correct instruction = wrong instruction = 0`, это эффект пола: такой языковой контроль **не позволяет** сделать вывод, использует ли модель инструкцию.

Полная неизменяемая версия: `Predictions.md`.


In [ ]:
import math
import shutil
import subprocess
import sys
from importlib.metadata import version

import pandas as pd
import torch
import lerobot

from src import analysis, data, evaluation, methods, train_baselines
from src.methods import H1Config
from src.settings import BUDGETS, RESULTS, TARGETS

subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Expected Python 3.12, got {sys.version.split()[0]}")
if lerobot.__version__ != "0.6.1":
    raise RuntimeError(f"Expected LeRobot 0.6.1, got {lerobot.__version__}")
if not torch.__version__.startswith("2.11.0"):
    raise RuntimeError(f"Expected PyTorch 2.11.0, got {torch.__version__}")
if torch.version.cuda != "12.8":
    raise RuntimeError(f"Expected CUDA 12.8 PyTorch build, got {torch.version.cuda}")
if version("torchcodec") != "0.11.1":
    raise RuntimeError(f"Expected torchcodec 0.11.1, got {version('torchcodec')}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 2**30
print(
    f"Python {sys.version.split()[0]} | LeRobot {lerobot.__version__} | "
    f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | torchcodec {version('torchcodec')}"
)
print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB")
# The defaults below are intentionally conservative for an RTX 6000-class server.
# SmolVLA itself is substantially lighter, but H1/H3 add temporary tensors and
# reference weights. Refuse only clearly smaller cards instead of requiring 80 GB.
if vram_gb < 40:
    raise RuntimeError(
        "Default batches are validated for a >=40 GB GPU. "
        f"Detected {vram_gb:.1f} GB; lower SEEN_BATCH/TARGET_BATCH/H1_PRIOR_BATCH before continuing."
    )

SEEN_BATCH = 32
TARGET_BATCH = 8
H1_PRIOR_BATCH = 16
AUX_SEED = 7
EVAL_EPISODES = 20
EVAL_SEED = 10_000
SEEN_SANITY_EPISODES = 5

pd.set_option("display.max_columns", None)
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Данные и ранние проверки

`prepare_data()`:

1. скачивает `libero_90` и `libero_goal` из `nvidia/LIBERO_LeRobot_v3`;
2. атомарно переводит только канал гриппера: `g_env = 1 - 2*g_data`;
3. **не перекодирует видео**: используется `torchcodec` поверх исходных mp4;
4. декодирует реальный кадр из обеих частей до обучения;
5. находит целевые эпизоды по точному тексту и проверяет `episodes=...` для каждого K;
6. создаёт K-only представления со статистиками нормировки только первых K демонстраций;
7. пишет `data/target_manifest.csv`.

Затем проверка воспроизведением запускает одну настоящую демонстрацию каждой целевой задачи в среде. Любая ошибка здесь означает, что обучение запускать нельзя.


In [ ]:
data_info = data.prepare_data()
data_info

In [ ]:
smoke_rows = []
for task_id in TARGETS:
    result = data.replay_gripper_smoke(task_id, k=5, episode=0)
    smoke_rows.append(result)

replay_smoke = pd.DataFrame(smoke_rows)
replay_smoke.to_csv(RESULTS / "replay_smoke.csv", index=False)
display(replay_smoke)

if not replay_smoke["success"].all():
    raise RuntimeError("At least one demonstration replay failed; stop before training.")

## 2. Единый бюджет шагов

Все методы получают одинаковое число **целевых** обновлений в одной и той же клетке `(task, K)`. Это исключает скрытую фору одному из методов. H4 дополнительно делает обновление по seen-батчу каждый четвёртый шаг; эта вычислительная цена учитывается отдельно.


In [ ]:
seen_steps = math.ceil(2.0 * data_info["seen_train_frames"] / SEEN_BATCH)

target_steps = {}
for task_id in TARGETS:
    for k in BUDGETS:
        frames = data_info["target_frames"][f"t{task_id}_k{k}"]
        target_samples = max(6400, math.ceil(2.5 * frames))
        target_steps[(task_id, k)] = math.ceil(target_samples / TARGET_BATCH)

schedule = pd.DataFrame([
    {
        "task_id": task_id,
        "instruction": TARGETS[task_id],
        "K": k,
        "frames": data_info["target_frames"][f"t{task_id}_k{k}"],
        "target_steps": target_steps[(task_id, k)],
    }
    for task_id in TARGETS
    for k in BUDGETS
])

print("seen steps:", seen_steps)
display(schedule)

## 3. Seen-претрен и обязательная проверка

Сначала обучается seen-чекпойнт. После этого автоматически выбираются три benchmark task id из `libero_90`, чьи тексты реально присутствуют в датасете, и выполняется по 5 эпизодов.

**Защитное правило:** если все 15 seen-эпизодов дают ноль, запуск останавливается. Это отделяет вероятную проблему обучения/оценки от допустимого нулевого переноса на новые `libero_goal`.


In [ ]:
seen = train_baselines.train_seen(
    steps=seen_steps,
    batch_size=SEEN_BATCH,
    seed=AUX_SEED,
)
print("seen checkpoint:", seen)

In [ ]:
seen_eval_json, seen_tasks = evaluation.eval_seen_sanity(
    seen,
    n_tasks=3,
    n_episodes=SEEN_SANITY_EPISODES,
    seed=9_000,
)
seen_sanity = analysis.collect_eval_results([
    {
        "eval_json": str(seen_eval_json),
        "method": "seen_sanity",
        "budget": -1,
        "train_seed": AUX_SEED,
    }
])
seen_text = dict(seen_tasks)
seen_sanity["instruction"] = seen_sanity["task_id"].map(seen_text)
seen_sanity.to_csv(RESULTS / "seen_sanity.csv", index=False)
display(
    seen_sanity.groupby(["task_id", "instruction"], as_index=False)
    .agg(successes=("success", "sum"), episodes=("success", "size"))
)

if int(seen_sanity["success"].sum()) == 0:
    raise RuntimeError(
        "Seen checkpoint scored 0/15 on LIBERO-90 sanity tasks. "
        "Do not spend compute on the target grid; inspect outputs/logs and the seen checkpoint first."
    )

## 4. K=0 и контроль языка

K=0 — seen-чекпойнт на трёх целевых задачах без целевого дообучения. Затем те же task/seed пары оцениваются с циклически подменённой целевой инструкцией. Если обе стороны находятся на нуле, результат сохраняется, но не интерпретируется как отсутствие языковой чувствительности.


In [ ]:
zero_json = evaluation.eval_checkpoint(
    seen,
    "zero_shot",
    tuple(TARGETS),
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
    keep_failed_videos=1,
)
zero_raw = analysis.collect_eval_results([
    {
        "eval_json": str(zero_json),
        "method": "zero_shot",
        "budget": 0,
        "train_seed": float("nan"),
    }
])
wrong_raw = evaluation.eval_wrong_language(
    seen,
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)
language_raw = analysis.language_control(zero_raw, wrong_raw, start_seed=EVAL_SEED)
language_summary = analysis.language_control_summary(language_raw)
language_raw.to_csv(RESULTS / "language_control.csv", index=False)
language_summary.to_csv(RESULTS / "language_control_summary.csv", index=False)
display(language_summary)

## 5. Предварительная проверка H1–H4

До запуска 18 обучений базового метода один раз строится H1-представление из seen-видео и выполняется **один настоящий forward/backward без шага оптимизатора** для H1, H2, H3 и H4 на K=5. Эта проверка ловит несовместимость API, размерностей, процессоров, hook на скрытые токены и вспомогательных функций до дорогой сетки. H1-представление затем переиспользуется в основном эксперименте.


In [ ]:
h1_prior = methods.build_h1_video_prior(
    seen,
    batch_size=H1_PRIOR_BATCH,
    seed=AUX_SEED,
    config=H1Config(horizon=10, clusters=64, max_pairs=4096, kmeans_iters=15),
)
preflight = methods.smoke_custom_objectives(
    seen,
    h1_prior,
    task_id=0,
    k=5,
    batch_size=2,
    seed=31_415,
)
print("custom objectives smoke: OK")
display(pd.DataFrame([preflight]))


## 6. Базовый метод

Штатное дообучение SmolVLA на первых 5/10/25 демонстрациях каждой целевой задачи, два сида обучения. После обучения сразу выполняется обязательная оценка: так аномальный baseline обнаружится до запуска четырёх дополнительных методов.


In [ ]:
baseline_checkpoints = train_baselines.train_baseline_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
)
baseline_specs = evaluation.evaluate_grid(
    baseline_checkpoints,
    "baseline",
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)
baseline_raw = analysis.collect_eval_results(baseline_specs)
display(analysis.cost_curve_summary(
    analysis.attach_shared_zero_shot(
        pd.concat([zero_raw, baseline_raw], ignore_index=True),
        ("baseline",),
    )
))

## 7. H1 — динамика по видео / Bonus A

Сначала строятся 64 прототипа визуальных переходов максимум по 4096 парам кадров `libero_90`. Значения действий удаляются из батча до препроцессинга и не участвуют в этой стадии. Затем скрытое состояние целевой политики получает дополнительную задачу предсказать ближайший прототип и направление наблюдаемого перехода.


In [ ]:
# h1_prior уже построен и проверен в предварительной проверке выше.
h1_checkpoints = methods.train_h1_grid(
    seen,
    h1_prior,
    target_steps,
    batch_size=TARGET_BATCH,
    lambda_dyn=0.10,
    cosine_weight=0.25,
)
h1_specs = evaluation.evaluate_grid(
    h1_checkpoints, "H1", n_episodes=EVAL_EPISODES, seed=EVAL_SEED
)
h1_raw = analysis.collect_eval_results(h1_specs)

## 8. H2 — агрегация вниманием по инструкции

Правильная инструкция формирует запрос внимания к скрытым токенам эксперта действий. Для посторонней инструкции используется только текст из seen-набора; данные других целевых задач не подмешиваются.


In [ ]:
h2_checkpoints = methods.train_h2_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
    lambda_instruction=0.10,
    margin=0.10,
)
h2_specs = evaluation.evaluate_grid(
    h2_checkpoints, "H2", n_episodes=EVAL_EPISODES, seed=EVAL_SEED
)
h2_raw = analysis.collect_eval_results(h2_specs)

## 9. H3 — L2-SP

Основная функция потерь дополняется средним квадратом смещения штатно обучаемых параметров от seen-чекпойнта. Вспомогательная голова не нужна, архитектура при оценке не меняется.


In [ ]:
h3_checkpoints = methods.train_h3_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
    lambda_sp=10_000.0,
)
h3_specs = evaluation.evaluate_grid(
    h3_checkpoints, "H3", n_episodes=EVAL_EPISODES, seed=EVAL_SEED
)
h3_raw = analysis.collect_eval_results(h3_specs)

## 10. H4 — подмешивание демонстраций из seen-части

Каждый четвёртый целевой шаг добавляется BC-loss на отдельном seen-батче. Бюджет целевой задачи остаётся ровно K. Оба батча нормируются статистиками первых K целевых демонстраций, чтобы одна выходная голова не обучалась в двух разных системах координат.


In [ ]:
h4_checkpoints = methods.train_h4_grid(
    seen,
    target_steps,
    batch_size=TARGET_BATCH,
    replay_interval=4,
    lambda_seen=0.5,
)
h4_specs = evaluation.evaluate_grid(
    h4_checkpoints, "H4", n_episodes=EVAL_EPISODES, seed=EVAL_SEED
)
h4_raw = analysis.collect_eval_results(h4_specs)

## 11. Итоговые таблицы

Автоматически сохраняются только численные таблицы и ссылки на небольшой набор видео с неудачами. Графики здесь не строятся: их можно добавить в GitHub после окончательного выбора визуализации.

`Summary.docx` при заполнении результатов ссылается на эти CSV, поэтому исходные числа остаются проверяемыми.


In [ ]:
all_nonzero = pd.concat(
    [baseline_raw, h1_raw, h2_raw, h3_raw, h4_raw],
    ignore_index=True,
)
combined = analysis.attach_shared_zero_shot(
    pd.concat([zero_raw, all_nonzero], ignore_index=True),
    ("baseline", "H1", "H2", "H3", "H4"),
)

rates = analysis.success_rates(combined)
cost_curve = analysis.cost_curve_summary(combined)
failures = analysis.failure_candidates(combined)

combined.to_csv(RESULTS / "evaluation.csv", index=False)
rates.to_csv(RESULTS / "success_rates.csv", index=False)
cost_curve.to_csv(RESULTS / "cost_curve.csv", index=False)
failures.to_csv(RESULTS / "failure_candidates.csv", index=False)

display(cost_curve)
display(rates)
print("saved:", sorted(str(p) for p in RESULTS.glob("*.csv")))
print("failure videos retained:", len(failures))